In [2]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Available GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
Available GPU: []


In [12]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.21.0
Keras: 3.13.2


In [1]:
from google.colab import files

uploaded = files.upload()

Saving cifar-10-python.tar.gz to cifar-10-python.tar.gz


In [2]:
import os
import tarfile

os.makedirs("data", exist_ok=True)

with tarfile.open("cifar-10-python.tar.gz", "r:gz") as tar:
    tar.extractall("data")

print("Dataset extracted successfully!")

print(os.listdir("data/cifar-10-batches-py"))

/tmp/ipykernel_420/3232461212.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("data")


Dataset extracted successfully!
['data_batch_2', 'readme.html', 'data_batch_4', 'batches.meta', 'test_batch', 'data_batch_3', 'data_batch_1', 'data_batch_5']


In [3]:
import os
import pickle
import numpy as np

DATASET_PATH = "data/cifar-10-batches-py"


def load_batch(file_path):

    with open(file_path, "rb") as file:
        batch = pickle.load(file, encoding="bytes")

    images = batch[b"data"]
    labels = batch[b"labels"]

    # Convert images into RGB format
    images = images.reshape(-1, 3, 32, 32)
    images = images.transpose(0, 2, 3, 1)

    return images, np.array(labels)


# Load training batches
X_train = []
y_train = []

for i in range(1, 6):

    images, labels = load_batch(
        os.path.join(DATASET_PATH, f"data_batch_{i}")
    )

    X_train.append(images)
    y_train.extend(labels)

X_train = np.concatenate(X_train)
y_train = np.array(y_train)


# Load testing batch
X_test, y_test = load_batch(
    os.path.join(DATASET_PATH, "test_batch")
)


# Display dataset information
print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)

print("Testing images:", X_test.shape)
print("Testing labels:", y_test.shape)

print("Number of classes:", len(np.unique(y_train)))

Training images: (50000, 32, 32, 3)
Training labels: (50000,)
Testing images: (10000, 32, 32, 3)
Testing labels: (10000,)
Number of classes: 10


In [4]:
import tensorflow as tf

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMAGE_SIZE = (96, 96)
BATCH_SIZE = 16


# Data augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.04),
    tf.keras.layers.RandomTranslation(
        height_factor=0.1,
        width_factor=0.1
    )
])


def prepare_dataset(images, labels, training=False):

    dataset = tf.data.Dataset.from_tensor_slices(
        (images, labels)
    )

    if training:
        dataset = dataset.shuffle(5000)

    def preprocess_image(image, label):

        image = tf.cast(image, tf.float32)

        image = tf.image.resize(
            image, IMAGE_SIZE
        )

        image = preprocess_input(image)

        return image, label

    dataset = dataset.map(
        preprocess_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        dataset = dataset.map(
            lambda image, label: (
                data_augmentation(image, training=True),
                label
            ),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_dataset = prepare_dataset(
    X_train, y_train, training=True
)

test_dataset = prepare_dataset(
    X_test, y_test, training=False
)


print("Preprocessing completed!")

for images, labels in train_dataset.take(1):

    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image data type:", images.dtype)

Preprocessing completed!
Image batch shape: (16, 96, 96, 3)
Label batch shape: (16,)
Image data type: <dtype: 'float32'>


In [5]:
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2

from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dense,
    Dropout,
    RandomFlip,
    RandomRotation,
    RandomTranslation
)

from tensorflow.keras.models import Model


def build_model():

    inputs = Input(shape=(96, 96, 3))

    # Data augmentation
    x = RandomFlip("horizontal")(inputs)
    x = RandomRotation(0.04)(x)

    x = RandomTranslation(
        height_factor=0.1,
        width_factor=0.1
    )(x)

    # Load pretrained MobileNetV2
    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(96, 96, 3),
        name="mobilenetv2_base"
    )

    # Enable direct fine-tuning
    base_model.trainable = True

    # Freeze earlier layers
    for layer in base_model.layers[:-30]:
        layer.trainable = False

    # Freeze Batch Normalization layers
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    # Pretrained feature extraction during training
    # Selected trainable weights are still updated by backpropagation
    x = base_model(x, training=False)

    # Classification head
    x = GlobalAveragePooling2D()(x)

    x = Dense(
        128,
        activation="relu"
    )(x)

    x = Dropout(0.3)(x)

    outputs = Dense(
        10,
        activation="softmax"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs
    )

    return model


model = build_model()

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 96, 96, 3)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation_1            │ (None, 96, 96, 3)      │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_base (Functional)   │ (None, 3, 3, 1280)     │     2,257,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,242 (9.24 MB)

 Trainable params: 1,675,978 (6.39 MB)

 Non-trainable params: 747,264 (2.85 MB)

In [6]:
print("Total parameters:", model.count_params())

print(
    "Trainable parameters:",
    sum(
        tf.size(variable).numpy()
        for variable in model.trainable_variables
    )
)

print(
    "Non-trainable parameters:",
    sum(
        tf.size(variable).numpy()
        for variable in model.non_trainable_variables
    )
)

Total parameters: 2423242
Trainable parameters: 1675978
Non-trainable parameters: 747272


In [7]:
import os
import tensorflow as tf

# Create folder to save trained models
os.makedirs("models", exist_ok=True)

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

Model compiled successfully!


In [8]:
callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath="models/best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=1,
        min_lr=0.0000001,
        verbose=1
    )
]

print("Callbacks configured successfully!")

Callbacks configured successfully!


In [9]:
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=5,
    callbacks=callbacks
)

Epoch 1/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.5176 - loss: 1.4062
Epoch 1: val_accuracy improved from None to 0.81360, saving model to models/best_model.keras

Epoch 1: finished saving model to models/best_model.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 217s 65ms/step - accuracy: 0.6368 - loss: 1.0601 - val_accuracy: 0.8136 - val_loss: 0.5552 - learning_rate: 1.0000e-05
Epoch 2/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.7465 - loss: 0.7470
Epoch 2: val_accuracy improved from 0.81360 to 0.84420, saving model to models/best_model.keras

Epoch 2: finished saving model to models/best_model.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 199s 64ms/step - accuracy: 0.7531 - loss: 0.7242 - val_accuracy: 0.8442 - val_loss: 0.4735 - learning_rate: 1.0000e-05
Epoch 3/5
3124/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.7769 - loss: 0.6547
Epoch 3: val_accuracy improved from 0.84420 to 0.84790, saving model to models/best_model.keras

Epoch 3: finished saving model t

In [13]:
model.save_weights("best_model.weights.h5")

In [14]:
from google.colab import files

files.download("best_model.weights.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>